In [34]:
import pandas as pd
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from dotenv import load_dotenv
import os

In [35]:
load_dotenv()

BASE_PATH = os.getenv("BASE_PATH")
X_train = pd.read_parquet(f"{BASE_PATH}/data/final/X_train_final.parquet")
X_test = pd.read_parquet(f"{BASE_PATH}/data/final/X_test_final.parquet")
y_train = pd.read_parquet(f"{BASE_PATH}/data/final/y_train_final.parquet")
y_test = pd.read_parquet(f"{BASE_PATH}/data/final/y_test_final.parquet")

In [36]:
y_train = y_train.iloc[:, 0]
y_test = y_test.iloc[:, 0]

In [37]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(y_train.dtypes, type(y_train))

(79588, 126) (19898, 126) (79588,) (19898,)
bool <class 'pandas.Series'>


In [38]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
for col in [X_tr, X_val, y_tr, y_val]:
    print(col.shape)

(63670, 126)
(15918, 126)
(63670,)
(15918,)


In [39]:
neg, pos = y_tr.value_counts()[False], y_tr.value_counts()[True]
scale_pos_weight = neg / pos

print(f"Negative (no rain): {neg}, Positive (rain): {pos}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

Negative (no rain): 49362, Positive (rain): 14308
scale_pos_weight: 3.4500


In [40]:
model = lgb.LGBMClassifier(
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_estimators=1000,
    metric='average_precision'
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]
)

/home/youssef/Projects/aus-weather-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 14308, number of negative: 49362
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004020 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5302
[LightGBM] [Info] Number of data points in the train set: 63670, number of used features: 126
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.224721 -> initscore=-1.238362
[LightGBM] [Info] Start training from score -1.238362
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.727803
[100]	valid_0's average_precision: 0.73925
[150]	valid_0's average_precision: 0.741709
[200]	valid_0's average_precision: 0.743315
[250]	valid_0's average_precision: 0.744584
[300]	valid_0's average_precision: 0.745572
[350]	valid_0's average_precision: 0.745171
Early stopping, best iteration is:
[302]	valid_0's average_precision: 0.74

,n_estimators,1000
,objective,'binary'
,random_state,42
,scale_pos_weight,np.float64(3.449958065417948)
,metric,'average_precision'
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,class_weight,None


In [41]:
y_pred_proba = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"PR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

PR-AUC: 0.7553
ROC-AUC: 0.8933


In [ ]:
import joblib
joblib.dump(model, f"{BASE_PATH}/models/lgbm_baseline.pkl")